# Stablecoin StressBench — Feature Engineering

This notebook demonstrates the feature engineering pipeline:
- Microstructure features (spread, depth, imbalance)
- Cross-quote basis computation
- Fragmentation index
- Settlement proxy features
- Label distributions

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from stressbench.book.order_book import OrderBook
from stressbench.book.vwap import executable_buy_vwap, executable_sell_vwap, net_profit_bps
from stressbench.features.basis import (
    compute_stablecoin_usd_price,
    compute_cross_quote_basis_bps,
    compute_fragmentation_features,
)
from stressbench.features.microstructure import compute_book_snapshot_features

print('Imports OK')

## 1. Order Book Reconstruction Demo

In [ ]:
# Construct a synthetic order book
book = OrderBook()
book.apply_snapshot(
    bids=[('0.9999', '50000'), ('0.9998', '100000'), ('0.9997', '200000')],
    asks=[('1.0001', '30000'), ('1.0002', '80000'), ('1.0003', '150000')],
)

print(f'Best bid: {book.best_bid()}')
print(f'Best ask: {book.best_ask()}')
print(f'Mid: {book.mid()}')
print(f'Spread: {book.spread():.6f}')
print(f'Spread (bps): {book.spread_bps():.2f}')
print(f'Imbalance (1bp): {book.imbalance(1.0):.4f}')

## 2. Executable VWAP at Different Sizes

In [ ]:
sizes = [10_000, 50_000, 100_000, 200_000, 300_000]
buy_vwaps = [executable_buy_vwap(book, q) for q in sizes]
sell_vwaps = [executable_sell_vwap(book, q) for q in sizes]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sizes, buy_vwaps, 'r-o', label='Buy VWAP')
ax.plot(sizes, sell_vwaps, 'b-o', label='Sell VWAP')
ax.axhline(book.mid(), color='gray', ls='--', label='Mid')
ax.set_xlabel('Quantity (USD)')
ax.set_ylabel('Executable Price')
ax.set_title('Executable VWAP vs Notional Size')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Cross-Quote Basis Demo

In [ ]:
# Simulate BTC/USD direct vs BTC/USDC implied
btc_usd_direct = 50000.0
usdc_usd = 0.9998  # USDC is slightly below peg
btc_usdc = btc_usd_direct / usdc_usd  # BTC/USDC price
btc_usd_via_usdc = btc_usdc * usdc_usd  # Back to USD

basis = compute_cross_quote_basis_bps(btc_usd_direct, btc_usd_via_usdc)
print(f'BTC/USD direct: {btc_usd_direct}')
print(f'USDC/USD: {usdc_usd}')
print(f'BTC/USDC: {btc_usdc:.2f}')
print(f'Cross-quote basis: {basis:.2f} bps')